In [1]:
import pandas as pd
import csv
import pickle
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import os
import random
from itertools import product


from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search

In [ ]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)

In [ ]:
random.seed(42)
# create feature matrix
X = feature_matrix.copy()
random.seed(42)

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# # User-selected number of each (or use len(...) for "all")
# num_topics = len(topic_cols)  # e.g., 100
# num_stocks = 0

# # Randomly sample (without replacement), limited by available count
# selected_topics = random.sample(topic_cols, min(num_topics, len(topic_cols)))
# selected_stocks = random.sample(stock_cols, min(num_stocks, len(stock_cols)))

# # Final filtered dataframe
# X = X[selected_topics + selected_stocks]

# # Convert stock returns to log returns: log(1+r)
# X[selected_stocks] = np.log(X[selected_stocks] + 1)

# # ensure that all indices align
# common_index = X.index.intersection(y.index)
# X = X.loc[common_index]
# y = y.loc[common_index]

In [ ]:
# create featrue matrix containing only stocks
y_stocks = feature_matrix[stock_cols]
X_topics = feature_matrix[topic_cols]

In [9]:
import pandas as pd
import numpy as np
from grid_search import estimate_single_config

def run_first_100_y_columns(
    X_stocks, X_topics, window_size=350, n_lags=8, lambda_val=0.0021784
):
    all_summaries = []
    all_details = []

    for i in range(200):
        print(f"Running column {i}...")

        y = X_stocks.iloc[:, i]
        result = estimate_single_config(
            X=X_topics,
            y=y,
            window_size=window_size,
            n_lags=n_lags,
            lambda_val=lambda_val
        )

        summary = pd.DataFrame([result['summary']])
        summary['target_column'] = X_stocks.columns[i]
        all_summaries.append(summary)

        details = result['details'].copy()
        details['target_column'] = X_stocks.columns[i]
        all_details.append(details)

    summaries_df = pd.concat(all_summaries, ignore_index=True)
    details_df = pd.concat(all_details, ignore_index=True)

    return summaries_df, details_df


In [10]:
summaries, details = run_first_100_y_columns(X_stocks, X_topics)

print("All Summaries:")
print(summaries.head())

print("\nAll Details:")
print(details.head())

Running column 0...
Running column 1...
Running column 2...
Running column 3...
Running column 4...
Running column 5...
Running column 6...
Running column 7...
Running column 8...
Running column 9...
Running column 10...
Running column 11...
Running column 12...
Running column 13...
Running column 14...
Running column 15...
Running column 16...
Running column 17...
Running column 18...
Running column 19...
Running column 20...
Running column 21...


c:\Users\jonat\Lasso_paper\Empirical\scripts\lasso_11_2025\stage2.py:23: RuntimeWarning: invalid value encountered in log
  return np.log(1 - kappa * np.exp(pred_t)) - np.log(1 - kappa * np.exp(pred_t1)) + intercept


Running column 22...
Running column 23...
Running column 24...
Running column 25...
Running column 26...
Running column 27...
Running column 28...
Running column 29...
Running column 30...
Running column 31...
Running column 32...
Running column 33...
Running column 34...
Running column 35...
Running column 36...
Running column 37...
Running column 38...
Running column 39...
Running column 40...
Running column 41...
Running column 42...
Running column 43...
Running column 44...
Running column 45...
Running column 46...
Running column 47...
Running column 48...
Running column 49...
Running column 50...
Running column 51...
Running column 52...


c:\Users\jonat\Lasso_paper\Empirical\scripts\lasso_11_2025\stage2.py:23: RuntimeWarning: invalid value encountered in log
  return np.log(1 - kappa * np.exp(pred_t)) - np.log(1 - kappa * np.exp(pred_t1)) + intercept


Running column 53...
Running column 54...
Running column 55...
Running column 56...
Running column 57...
Running column 58...
Running column 59...
Running column 60...
Running column 61...
Running column 62...
Running column 63...
Running column 64...
Running column 65...
Running column 66...
Running column 67...
Running column 68...
Running column 69...
Running column 70...
Running column 71...
Running column 72...
Running column 73...
Running column 74...
Running column 75...
Running column 76...
Running column 77...
Running column 78...
Running column 79...
Running column 80...
Running column 81...
Running column 82...
Running column 83...
Running column 84...


...

Running column 145...
Running column 146...
Running column 147...
Running column 148...
Running column 149...
Running column 150...
Running column 151...
Running column 152...
Running column 153...
Running column 154...
Running column 155...
Running column 156...
Running column 157...
Running column 158...
Running column 159...
Running column 160...
Running column 161...
Running column 162...
Running column 163...
Running column 164...
Running column 165...
Running column 166...
Running column 167...
Running column 168...
Running column 169...
Running column 170...
Running column 171...
Running column 172...
Running column 173...
Running column 174...
Running column 175...
Running column 176...
Running column 177...
Running column 178...
Running column 179...
Running column 180...
Running column 181...
Running column 182...
Running column 183...
Running column 184...
Running column 185...
Running column 186...
Running column 187...


c:\Users\jonat\Lasso_paper\Empirical\scripts\lasso_11_2025\stage2.py:23: RuntimeWarning: invalid value encountered in log
  return np.log(1 - kappa * np.exp(pred_t)) - np.log(1 - kappa * np.exp(pred_t1)) + intercept


Running column 188...
Running column 189...
Running column 190...
Running column 191...
Running column 192...
Running column 193...
Running column 194...
Running column 195...
Running column 196...
Running column 197...
Running column 198...
Running column 199...


C:\Users\jonat\AppData\Local\Temp\ipykernel_33856\1521774919.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  details_df = pd.concat(all_details, ignore_index=True)


All Summaries:
   window_size  n_lags    lambda  r2_insample_stage1  r2_oos_stage1  \
0          350       8  0.002178            0.000746      -0.002494   
1          350       8  0.002178            0.040449      -0.012617   
2          350       8  0.002178            0.232730      -0.052130   
3          350       8  0.002178            0.119384      -0.032220   
4          350       8  0.002178            0.055244      -0.005476   

   r2_insample_stage2  r2_oos_stage2     kappa  kappa_tstat  intercept  \
0            0.001600       0.012558  0.464878     1.361699   0.000485   
1            0.002836      -0.005609  0.142773     1.079179   0.000716   
2           -0.000889      -0.013570  0.090428     1.357813   0.000911   
3           -0.005108      -0.016862  0.259081     4.165668   0.000293   
4            0.000528      -0.001252  0.048350     0.293736   0.000402   

   intercept_tstat  n_observations  n_windows  n_oos_predictions_stage2  \
0         2.135446            1532    

In [12]:
pd.DataFrame.to_parquet(summaries, 'stage1_stage2_summaries.parquet')
pd.DataFrame.to_parquet(details, 'stage1_stage2_details.parquet')

In [ ]:
summaries = pd.read_parquet('stage1_stage2_summaries.parquet')
details = pd.read_parquet('stage1_stage2_details.parquet')